# Watermark Predictor Benchmark Results

Plots for `results/results.csv`. Hue is always the `predictor`. VLDB/SIGMOD-style figures.

Run `pip install pandas matplotlib seaborn` if not already available.

## Reading the `trace` labels

Each trace is a synthetic watermark stream — a sequence of `(watermark value, wall-clock arrival)` **samples**. A label combines a base regime with optional noise:

**Base regime** (the clean signal — deterministic, no randomness):
- `ConstantRate(r)` — single phase, watermark advances at a fixed rate `r` the whole time.
- `RateChange(1->4)` — rate steps **up** at a phase boundary (1.0 → 4.0).
- `Stall(2.0->0)` — the watermark **holds flat** (rate 0) while wall-clock keeps advancing, then resumes. Models an idle source / event-time gap.
- `CatchUp(2.0->8.0)` — a short stall followed by a **fast burst** (rate 8.0) that drains the backlog before settling. Breaks predictors that carried the stalled rate forward.

**`+N samples`** — appears on traces with a regime change (`RateChange`, `Stall`, `CatchUp`). `N` = how many samples the predictor observes *after* the onset of the change before it must predict. It's the **convergence horizon**: `+2` = predict almost immediately (predictor hasn't caught up yet), `+50` = lots of time to re-adapt. `ConstantRate` is steady-state, so it never carries this tag.

**Noise tags** — perturb only the *wall-clock arrival* of each sample; the watermark values themselves are untouched:
- `clean` — no noise.
- `mild/heavy jitter (sd=σ)` — zero-mean Gaussian jitter on arrivals (σ = 10 / 30). Symmetric.
- `X% stragglers` — a fraction `X` of samples get an extra one-sided delay (heavy positive tail), on top of light jitter.

Tags compose: `RateChange(1->4) +20 + heavy jitter` = a rate step, 20 post-change samples observed, *and* noisy arrivals. `ConstantRate(2.0) + 10% stragglers` has only the noise axis (no `+N`, since there's no rate change to converge to).

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

# results.csv is LONG now: one row per scored prediction from the rolling (prequential) evaluation.
#   eval_offset    = samples since warm-up ended (0 = first online prediction; the regime onset for event traces)
#   horizon        = event-time look-ahead of the query (think window size)
#   abs_err        = |predicted - true| wall-clock
#   signed_err     = predicted - true wall-clock (sign = early / late)
#   true_wall      = ground-truth crossing wall-clock (MAPE denominator)
#   ns_per_predict = per-predict latency for the cell, denormalised onto every row
df = pd.read_csv(Path('results') / 'results.csv')

# --- parse the trace string into structured scenario columns ------------
def family(t):
    if t.startswith('ConstantRate'):
        return 'ConstantRate'
    if t.startswith('Stall'):
        return 'Stall'
    if t.startswith('CatchUp'):
        return 'CatchUp'
    return 'Other'

def noise(t):
    if 'heavy jitter' in t:
        return 'heavy jitter'
    if 'mild jitter' in t:
        return 'mild jitter'
    if 'heavy stragglers' in t:
        return 'heavy stragglers'
    if 'stragglers' in t:
        return 'stragglers'
    return 'clean'

def grab(pattern, t, cast=float):
    m = re.search(pattern, t)
    return cast(m.group(1)) if m else None

df['family'] = df['trace'].map(family)
df['noise'] = df['trace'].map(noise)
df['straggler_pct'] = df['trace'].apply(lambda t: grab(r'(\d+)% ', t, int))   # heavy-straggler fraction
df['jitter_sd'] = df['trace'].apply(lambda t: grab(r'sd=(\d+)', t, int))       # jitter standard deviation

# --- aggregate the long frame to one metrics row per (trace, predictor) -----
# Headline accuracy views (boxplots, heatmap, latency, sweeps) use `cell`; the rolling/horizon
# curves use the long `df` directly. MAPE skips true_wall==0 rather than blowing up.
def _agg(g):
    ae, se, tw = g['abs_err'], g['signed_err'], g['true_wall']
    m = tw > 0
    return pd.Series({
        'mae': ae.mean(),
        'rmse': np.sqrt((se ** 2).mean()),
        'mape_pct': (ae[m] / tw[m] * 100).mean() if m.any() else np.nan,
        'max_err': ae.max(),
        'samples': len(g),
        'ns_per_predict': g['ns_per_predict'].iloc[0],
    })

cell = (df.groupby(['trace', 'predictor'], sort=False)[['abs_err', 'signed_err', 'true_wall', 'ns_per_predict']]
          .apply(_agg).reset_index())
for col in ['family', 'noise', 'straggler_pct', 'jitter_sd']:
    cell[col] = cell['trace'].map(df.drop_duplicates('trace').set_index('trace')[col])

PREDICTOR_ORDER = sorted(df['predictor'].unique())
print('long rows:', df.shape, '| cells:', cell.shape, '| predictors:', list(df['predictor'].unique()))
cell.head()

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

# VLDB / SIGMOD camera-ready style: serif, compact, no chartjunk.
sns.set_theme(context='paper', style='whitegrid', font='serif')
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 11,
    'axes.labelsize': 11,
    'axes.titleweight': 'bold',
    'legend.fontsize': 9,
    'legend.frameon': False,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.linewidth': 0.5,
    'grid.alpha': 0.4,
})
# tab10 = the most-separated qualitative hues; the muted 'colorblind' set washed out the
# same-family predictors (3x EWMA, 2x Kalman, Robust). MARKERS adds a second visual channel so
# the line plots stay readable where two hues sit close — pass them via style= in lineplot.
PALETTE = dict(zip(PREDICTOR_ORDER, sns.color_palette('tab10', len(PREDICTOR_ORDER))))
MARKERS = dict(zip(PREDICTOR_ORDER, ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>'][:len(PREDICTOR_ORDER)]))

def legend_below(ax, ncol=None):
    ax.legend(title='predictor', loc='upper center', bbox_to_anchor=(0.5, -0.18),
              ncol=ncol or len(PREDICTOR_ORDER))
print('style ready, palette:', PALETTE)

style ready, palette: {'EWMA(alpha=0.3)': (0.12156862745098039, 0.4666666666666667, 0.7058823529411765), 'EWMA(alpha=0.5)': (1.0, 0.4980392156862745, 0.054901960784313725), 'EWMA(alpha=1.0)': (0.17254901960784313, 0.6274509803921569, 0.17254901960784313), 'Kalman(config1)': (0.8392156862745098, 0.15294117647058825, 0.1568627450980392), 'Kalman(config2)': (0.5803921568627451, 0.403921568627451, 0.7411764705882353), 'RobustAdaptiveKalman': (0.5490196078431373, 0.33725490196078434, 0.29411764705882354)}


## 0. Scenario shapes
The raw signal each predictor must extrapolate: watermark **event-time** against **wall-clock** for every clean base scenario. Slope = advancement rate, flat = stall, steep = catch-up burst. Exported by the benchmark binary to `results/traces.csv` (single source of truth — same definitions the accuracy runs use).

In [ ]:
traces = pd.read_csv(Path('results') / 'traces.csv')
scenarios = list(traces['scenario'].unique())
ncol = 2
nrow = -(-len(scenarios) // ncol)  # ceil
fig, axes = plt.subplots(nrow, ncol, figsize=(9, 3.0 * nrow), sharex=True)
for ax, scen in zip(axes.flat, scenarios):
    g = traces[traces['scenario'] == scen]
    ax.plot(g['wall_clock'], g['event_time'], lw=1.6, color='C0')
    ax.set_title(scen)
    ax.set_xlabel('wall-clock')
    ax.set_ylabel('watermark (event-time)')
for ax in axes.flat[len(scenarios):]:
    ax.set_visible(False)
fig.suptitle('Scenario shapes: watermark event-time vs wall-clock')
fig.tight_layout()
plt.show()

## 1. Accuracy per scenario family
Mean error (MAE) and relative error (MAPE) aggregated by scenario family — the headline "who is more accurate" view.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharex=True)
for ax, (metric, label) in zip(axes, [('mae', 'MAE (wall-clock)'), ('mape_pct', 'MAPE (%)')]):
    sns.boxplot(cell, x='family', y=metric, hue='predictor', hue_order=PREDICTOR_ORDER,
                palette=PALETTE, fliersize=2, linewidth=0.8, ax=ax)
    ax.set_xlabel('')
    ax.set_ylabel(label)
    # symlog: stall/catch-up errors are orders of magnitude above the clean families; linear hides them.
    ax.set_yscale('symlog')
    ax.set_ylim(bottom=0)
    ax.tick_params(axis='x', rotation=20)
    ax.get_legend().remove()
axes[0].legend(title='predictor', loc='upper left', ncol=1)
fig.suptitle('Prediction error by scenario family (each point = one trace, aggregated over the rolling eval)')
fig.tight_layout()
plt.show()

## 2. Rolling error after a stall / catch-up
The prequential view: prediction error at every step of the online phase, `eval_offset=0` being the regime onset (warm-up ends there). **Stall**: rate drops to 0 — does the predictor stop extrapolating forward? **CatchUp**: a stall then a rate-8 burst — does it track the surge or lag carrying the stalled rate? Lines are the mean abs error over the three horizons; the spike-and-decay shape *is* the adaptation curve the old `+N` sweeps only sampled at a few points.

In [ ]:
for fam, title in [('Stall', 'Stall(2.0$\\rightarrow$0)'), ('CatchUp', 'CatchUp(2.0$\\rightarrow$8.0)')]:
    sub = df[(df['family'] == fam) & (df['noise'] == 'clean')]
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    sns.lineplot(sub, x='eval_offset', y='abs_err', hue='predictor', hue_order=PREDICTOR_ORDER,
                 palette=PALETTE, errorbar=None, ax=ax)   # mean over horizons, no CI band
    ax.set_yscale('symlog')
    ax.set_xlabel('samples since warm-up  (0 = regime onset)')
    ax.set_ylabel('mean abs error (wall-clock)')
    ax.set_title(f'Rolling error after {title}')
    ax.legend(title='predictor', loc='upper right', ncol=2, fontsize=8)
    fig.tight_layout()
    plt.show()

## 2b. Look-ahead sensitivity (error vs. horizon)
Now that the rolling eval owns the time axis, `horizon` is a clean second axis: how does error grow as you ask the predictor to forecast further ahead (bigger window)? A constant-velocity model with a slightly-off rate estimate is fine near-term and drifts far-term, because rate error scales with look-ahead. Mean abs error over all traces and eval steps, per horizon.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
sns.lineplot(df, x='horizon', y='abs_err', hue='predictor', hue_order=PREDICTOR_ORDER,
             style='predictor', markers=MARKERS, dashes=False, palette=PALETTE, errorbar=None, ax=ax)
ax.set_xlabel('horizon (event-time look-ahead)')
ax.set_ylabel('mean abs error (wall-clock)')
ax.set_yscale('symlog')
ax.set_title('Look-ahead sensitivity: error vs. horizon')
ax.legend(title='predictor', loc='upper left', ncol=1, fontsize=8)
fig.tight_layout()
plt.show()

## 3. Robustness to stragglers
Error as the fraction of heavy stragglers grows (ConstantRate(2.0) base). Tests how each predictor tolerates delayed/late samples.

In [ ]:
strag = cell[cell['straggler_pct'].notna() & (cell['family'] == 'ConstantRate')].copy()
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for ax, (metric, label) in zip(axes, [('mae', 'MAE (wall-clock)'), ('rmse', 'RMSE (wall-clock)')]):
    sns.lineplot(strag, x='straggler_pct', y=metric, hue='predictor', hue_order=PREDICTOR_ORDER,
                 style='predictor', markers=MARKERS, dashes=False, palette=PALETTE, ax=ax)
    ax.set_xlabel('heavy straggler fraction (%)')
    ax.set_ylabel(label)
    ax.set_xlim(left=0)
    ax.set_yscale('symlog')  # error grows steeply with straggler fraction; log keeps the low end legible
    ax.set_ylim(bottom=0)
    ax.get_legend().remove()
axes[0].legend(title='predictor', loc='upper left')
fig.suptitle('Robustness to heavy stragglers (ConstantRate 2.0)')
fig.tight_layout()
plt.show()

## 4. Sensitivity to jitter
Error grouped by jitter level (clean / mild sd=10 / heavy sd=30), across the traces that carry jitter.

In [ ]:
jit_order = ['clean', 'mild jitter', 'heavy jitter']
jit = cell[cell['noise'].isin(jit_order) & (cell['family'] == 'ConstantRate')].copy()
jit['noise'] = pd.Categorical(jit['noise'], categories=jit_order, ordered=True)
fig, ax = plt.subplots(figsize=(6, 3.6))
# one cell per (noise, predictor) -> bars, not boxes.
sns.barplot(jit, x='noise', y='mae', hue='predictor', hue_order=PREDICTOR_ORDER, palette=PALETTE, ax=ax)
ax.set_xlabel('jitter level')
ax.set_ylabel('MAE (wall-clock)')
ax.set_title('Sensitivity to timestamp jitter')
ax.legend(title='predictor', loc='upper left', fontsize=8)
fig.tight_layout()
plt.show()

## 5. Accuracy vs. latency trade-off
Per-predict cost (ns) against achieved error. Cheap *and* accurate is bottom-left. Marker per scenario family.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), gridspec_kw={'width_ratios': [2, 1]})

sns.scatterplot(cell, x='ns_per_predict', y='mae', hue='predictor', hue_order=PREDICTOR_ORDER,
                style='family', palette=PALETTE, s=55, ax=axes[0])
axes[0].set_xlabel('latency (ns / predict)')
axes[0].set_ylabel('MAE (wall-clock)')
axes[0].set_yscale('symlog')
axes[0].set_title('Accuracy vs. latency')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=8)

# per-predict latency is constant per predictor (cell-level) -> one value each.
lat = cell.drop_duplicates('predictor')
sns.barplot(lat, x='predictor', y='ns_per_predict', hue='predictor', hue_order=PREDICTOR_ORDER,
            palette=PALETTE, legend=False, ax=axes[1])
axes[1].set_xlabel('')
axes[1].set_ylabel('ns / predict')
axes[1].set_title('Per-predict cost')
axes[1].tick_params(axis='x', rotation=20)
fig.tight_layout()
plt.show()

## 6. Per-trace error heatmap
Full breakdown: every trace × predictor, colored by MAE. The dense "where does each predictor win/lose" view.

In [ ]:
from matplotlib.patches import Rectangle

# per-trace x predictor heatmap, for every error metric (all lower = better), from the aggregated cells.
# Color is normalized PER ROW (each trace) so a few huge-magnitude traces (stall/catch-up) don't
# flatten every other row to one color. Annotations stay the real values; cyan outlines the best.
METRICS = [('mae', 'MAE', '.0f'), ('rmse', 'RMSE', '.0f'), ('mape_pct', 'MAPE (%)', '.1f'), ('max_err', 'max error', '.0f')]
fig, axes = plt.subplots(1, len(METRICS), figsize=(3.0 * len(METRICS) + 1, 0.32 * cell['trace'].nunique() + 1),
                         sharey=True)
for ax, (metric, label, fmt) in zip(axes, METRICS):
    pivot = cell.pivot_table(index='trace', columns='predictor', values=metric, sort=False)[PREDICTOR_ORDER]
    span = (pivot.max(axis=1) - pivot.min(axis=1)).replace(0, 1)
    norm = pivot.sub(pivot.min(axis=1), axis=0).div(span, axis=0)  # 0 = best (lowest) .. 1 = worst per row
    sns.heatmap(norm, annot=pivot, fmt=fmt, cmap='rocket_r', linewidths=0.4,
                annot_kws={'fontsize': 7}, cbar=False, vmin=0, vmax=1, ax=ax)
    for row, vals in enumerate(pivot.values):
        if vals.min() == vals.max():
            continue
        for col in (vals == vals.min()).nonzero()[0]:
            ax.add_patch(Rectangle((col, row), 1, 1, fill=False, edgecolor='#00d0ff', lw=2.0))
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(label)
fig.suptitle('Per-trace error by predictor (color = rank within row, outlined = lowest)')
fig.tight_layout()
plt.show()